In [1]:
# 데이터 준비
import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs, make_classification, make_regression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

In [ ]:
# 분류 데이터 생성
# X: 독립변수
# y: 종속변수
X, y = make_classification(n_samples=100, n_features=5, random_state=42)
print(X)
print(y)

[[-0.43066755  0.67287309 -0.72427983 -0.53963044 -0.65160035]
 [ 0.21164583 -0.84389686  0.53479393  0.82584805  0.68195297]
 [ 1.09267506  0.40910605  1.10009583 -0.94275087 -0.98150865]
 [ 1.51990078 -0.77336118  1.99805321  0.15513175 -0.3853136 ]
 [-0.45390127 -2.18347304  0.24472415  2.59123946 -0.48423407]
 [-1.46361184  0.37531604 -1.79532002  0.25415746 -1.24778318]
 [ 0.88948365  0.80742726  0.73019848 -1.28568005  0.13074058]
 [-1.11327862  1.89033108 -1.92487377 -1.5598485   0.18645431]
 [ 0.63174629 -0.88541844  1.02703224  0.68057323  0.54709738]
 [-0.88602706 -0.83311649 -0.7173148   1.31217492  0.44381943]
 [ 0.56372286 -1.47487037  1.15509316  1.35536951 -0.2176812 ]
 [-1.02754411 -0.32929388 -1.05383855  0.82600732 -0.05952536]
 [ 0.19375402 -0.96314239  0.55600276  0.96423311 -0.20219265]
 [-0.05396947  0.15985512 -0.11708689 -0.15013844  0.82541635]
 [ 0.96335953  0.65992405  0.86561977 -1.15806823  0.46210347]
 [-1.09939128 -0.47936995 -1.08324727  1.02255619 -0.92

In [4]:
X.shape

(100, 5)

In [ ]:
# Helper function to create DataFrame
def create_classification_data():
    X, y = make_classification(n_samples=100, n_features=5, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    # feature_1에 결측치를 추가 (10% 비율로)
    missing_indices = np.random.choice(df.index, size=int(len(df) * 0.1), replace=False)
    df.loc[missing_indices, 'feature_1'] = np.nan
    
    return df

In [ ]:
def create_regression_data():
    X, y = make_regression(n_samples=100, n_features=5, noise=0.1, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    for column in df.columns[:-1]:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # 이상치를 추가할 인덱스를 랜덤하게 선택
        outlier_indices = np.random.choice(df.index, size=5, replace=False)
        for idx in outlier_indices:
            df.at[idx, column] = np.random.uniform(upper_bound + 1, upper_bound + 10)

    return df

In [ ]:
def create_blobs_data():
    X, y = make_blobs(n_samples=100, n_features=5, centers=3, random_state=42)
    df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(X.shape[1])])
    df['target'] = y
    
    return df

In [ ]:
# Generate datasets for each problem type
classification_df = create_classification_data()
regression_df = create_regression_data()
blobs_df = create_blobs_data()

In [ ]:
print('=== Classification Generated Datasets ===')
print(classification_df.info())
print('\n=== regression Generated Datasets ===')
print(regression_df.head())
print('\n=== blobs Generated Datasets ===')
print(blobs_df.head())

In [ ]:
print('[전처리 전] 결측치 수:', classification_df.isna().sum())
print('[전처리 전] 결측치 수:', classification_df.isna().sum().sum())

classification_df['feature_1'] = classification_df['feature_1'].fillna(classification_df['feature_1'].mean())

print('[전처리 후] 결측치 수:', classification_df.isna().sum().sum())

In [ ]:
print('이상치 처리 전 결측치 수:', regression_df['feature_3'].isna().sum())

Q1 = regression_df['feature_3'].quantile(0.25)
Q3 = regression_df['feature_3'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

regression_df.loc[(regression_df['feature_3'] < lower_bound) | (regression_df['feature_3'] > upper_bound), 'feature_3'] = None

print('이상치 처리 후 결측치 수:', regression_df['feature_3'].isna().sum())

regression_df.dropna(subset=['feature_3'], inplace=True)

print('결측치 삭제:', regression_df['feature_3'].isna().sum())